In [ ]:
# ==============================================================================
# IMPORTS & CONFIGURATION
# ==============================================================================

import os
import numpy as np
import pandas as pd
import copy
import gc
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from sklearn.preprocessing import LabelEncoder

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, OneCycleLR
from torch.cuda.amp import GradScaler, autocast

from tqdm.auto import tqdm

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️ Device: {device}')
if device.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ==============================================================================
# CONFIGURATION
# ==============================================================================

@dataclass
class TrainingConfig:
    """Configuration centralisée pour l'entraînement."""
    # Paths
    data_dir: str = '../data' if os.path.exists('../data') else './data'
    model_dir: str = '../models' if os.path.exists('../models') or not os.path.exists('./models') else './models'
    submission_dir: str = '../submission'
    
    # Training parameters
    batch_size: int = 64
    epochs: int = 30
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    dropout: float = 0.3
    
    # Regularization
    label_smoothing: float = 0.1
    gradient_clip: float = 1.0
    
    # Early stopping
    patience: int = 7
    min_delta: float = 0.001
    
    # Cross-validation
    n_folds: int = 5
    use_cv: bool = True
    
    # Mixed precision
    use_amp: bool = True
    
    # EMA
    use_ema: bool = True
    ema_decay: float = 0.999
    
    def __post_init__(self):
        os.makedirs(self.model_dir, exist_ok=True)
        os.makedirs(self.submission_dir, exist_ok=True)

config = TrainingConfig()
print(f"📁 Data dir: {config.data_dir}")
print(f"📁 Model dir: {config.model_dir}")

In [ ]:
# ==============================================================================
# DATA LOADING
# ==============================================================================

def clean_array(arr: np.ndarray) -> np.ndarray:
    """Nettoie les NaN/Inf et convertit en float32."""
    arr = np.asarray(arr)
    arr = np.nan_to_num(arr, nan=0.0, posinf=1e6, neginf=-1e6)
    return arr.astype(np.float32)

# Paths
EMB_DIR = os.path.join(config.data_dir, 'embeddings')
FEAT_DIR = os.path.join(config.data_dir, 'features')

# Load embeddings
train_text_path = os.path.join(EMB_DIR, 'X_train_text_only_embeddings.npy')
train_desc_path = os.path.join(EMB_DIR, 'X_train_desc_only_embeddings.npy')
kaggle_text_path = os.path.join(EMB_DIR, 'X_kaggle_text_only_embeddings.npy')
kaggle_desc_path = os.path.join(EMB_DIR, 'X_kaggle_desc_only_embeddings.npy')

# Fallback to multilayer embeddings if available
multilayer_train = os.path.join(EMB_DIR, 'X_train_multilayer_embeddings.npy')
multilayer_kaggle = os.path.join(EMB_DIR, 'X_kaggle_multilayer_embeddings.npy')

if os.path.exists(multilayer_train):
    print("📦 Loading multilayer embeddings...")
    X_train_emb = clean_array(np.load(multilayer_train))
    X_kaggle_emb = clean_array(np.load(multilayer_kaggle))
    # Split into tweet and user (assuming equal split)
    half_dim = X_train_emb.shape[1] // 2
    X_train_t = X_train_emb[:, :half_dim]
    X_train_u = X_train_emb[:, half_dim:]
    X_kaggle_t = X_kaggle_emb[:, :half_dim]
    X_kaggle_u = X_kaggle_emb[:, half_dim:]
else:
    print("📦 Loading standard embeddings...")
    X_train_t = clean_array(np.load(train_text_path))
    X_train_u = clean_array(np.load(train_desc_path))
    X_kaggle_t = clean_array(np.load(kaggle_text_path))
    X_kaggle_u = clean_array(np.load(kaggle_desc_path))

# Load features
X_train_meta = clean_array(np.load(os.path.join(FEAT_DIR, 'X_train_features.npy')))
X_kaggle_meta = clean_array(np.load(os.path.join(FEAT_DIR, 'X_kaggle_features.npy')))

# Load labels
y = np.load(os.path.join(config.data_dir, 'y_train.npy'))
le = LabelEncoder()
y = le.fit_transform(y)

# Dimensions
tweet_dim = X_train_t.shape[1]
user_dim = X_train_u.shape[1]
meta_dim = X_train_meta.shape[1]
n_classes = len(le.classes_)

print(f"\n📊 Data shapes:")
print(f"   Tweet embeddings: {X_train_t.shape}")
print(f"   User embeddings: {X_train_u.shape}")
print(f"   Meta features: {X_train_meta.shape}")
print(f"   Labels: {y.shape} ({n_classes} classes)")
print(f"   Class distribution: {np.bincount(y)}")

In [ ]:
# ==============================================================================
# DATASET CLASS
# ==============================================================================

class InfluencerDataset(Dataset):
    """Dataset pour les données multi-modales."""
    
    def __init__(
        self, 
        tweet_emb: np.ndarray, 
        user_emb: np.ndarray, 
        meta: np.ndarray, 
        labels: Optional[np.ndarray] = None
    ):
        self.tweet = torch.from_numpy(tweet_emb).float()
        self.user = torch.from_numpy(user_emb).float()
        self.meta = torch.from_numpy(meta).float()
        self.labels = None if labels is None else torch.from_numpy(labels).long()
        
    def __len__(self) -> int:
        return self.tweet.shape[0]
    
    def __getitem__(self, idx: int):
        if self.labels is None:
            return self.tweet[idx], self.user[idx], self.meta[idx]
        return self.tweet[idx], self.user[idx], self.meta[idx], self.labels[idx]

In [ ]:
# ==============================================================================
# MODEL ARCHITECTURE (IMPROVED)
# ==============================================================================

class AttentionFusion(nn.Module):
    """Attention-based fusion pour combiner les modalités."""
    
    def __init__(self, hidden_dim: int, n_modalities: int = 3):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim * n_modalities, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, n_modalities),
            nn.Softmax(dim=-1)
        )
        self.n_modalities = n_modalities
        self.hidden_dim = hidden_dim
        
    def forward(self, *modalities: torch.Tensor) -> torch.Tensor:
        # Stack modalities: (B, n_modalities, hidden_dim)
        stacked = torch.stack(modalities, dim=1)
        
        # Compute attention weights
        concat = torch.cat(modalities, dim=-1)  # (B, n_modalities * hidden_dim)
        attn_weights = self.attention(concat)   # (B, n_modalities)
        
        # Weighted sum
        attn_weights = attn_weights.unsqueeze(-1)  # (B, n_modalities, 1)
        fused = (stacked * attn_weights).sum(dim=1)  # (B, hidden_dim)
        
        return fused, attn_weights.squeeze(-1)


class MultiSampleDropout(nn.Module):
    """Multi-sample dropout pour réduire la variance."""
    
    def __init__(self, dropout_rate: float = 0.3, n_samples: int = 5):
        super().__init__()
        self.dropouts = nn.ModuleList([nn.Dropout(dropout_rate) for _ in range(n_samples)])
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.training:
            return torch.mean(torch.stack([d(x) for d in self.dropouts]), dim=0)
        return x


class ImprovedInfluencerModel(nn.Module):
    """Modèle amélioré avec attention fusion et régularisation."""
    
    def __init__(
        self, 
        tweet_dim: int, 
        user_dim: int, 
        meta_dim: int, 
        n_classes: int,
        hidden_dim: int = 256,
        dropout: float = 0.3
    ):
        super().__init__()
        
        # Tweet branch
        self.tweet_branch = nn.Sequential(
            nn.Linear(tweet_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU()
        )
        
        # User branch
        self.user_branch = nn.Sequential(
            nn.Linear(user_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU()
        )
        
        # Meta branch (smaller due to fewer features)
        meta_hidden = min(hidden_dim // 2, meta_dim * 2)
        self.meta_branch = nn.Sequential(
            nn.Linear(meta_dim, meta_hidden),
            nn.BatchNorm1d(meta_hidden),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(meta_hidden, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU()
        )
        
        # Attention fusion
        self.fusion = AttentionFusion(hidden_dim, n_modalities=3)
        
        # Classification head with multi-sample dropout
        self.ms_dropout = MultiSampleDropout(dropout, n_samples=5)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, n_classes)
        )
        
        # Initialize weights
        self.apply(self._init_weights)
        
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.BatchNorm1d):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)
    
    def forward(
        self, 
        tweet: torch.Tensor, 
        user: torch.Tensor, 
        meta: torch.Tensor,
        return_attention: bool = False
    ) -> torch.Tensor:
        # Process each modality
        t = self.tweet_branch(tweet)
        u = self.user_branch(user)
        m = self.meta_branch(meta)
        
        # Attention-based fusion
        fused, attn_weights = self.fusion(t, u, m)
        
        # Classification
        x = self.ms_dropout(fused)
        logits = self.classifier(x)
        
        if return_attention:
            return logits, attn_weights
        return logits


# Test model
model = ImprovedInfluencerModel(tweet_dim, user_dim, meta_dim, n_classes, dropout=config.dropout)
model.to(device)

# Count parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n🧠 Model architecture:")
print(model)
print(f"\n📊 Trainable parameters: {n_params:,}")

In [ ]:
# ==============================================================================
# TRAINING UTILITIES
# ==============================================================================

class EarlyStopping:
    """Early stopping avec patience."""
    
    def __init__(self, patience: int = 7, min_delta: float = 0.001, mode: str = 'max'):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.counter = 0
        self.best_score = None
        self.should_stop = False
        
    def __call__(self, score: float) -> bool:
        if self.best_score is None:
            self.best_score = score
            return False
        
        if self.mode == 'max':
            improved = score > self.best_score + self.min_delta
        else:
            improved = score < self.best_score - self.min_delta
            
        if improved:
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
                
        return self.should_stop


class EMA:
    """Exponential Moving Average des poids du modèle."""
    
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        self.register()
        
    def register(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()
                
    def update(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = (
                    self.decay * self.shadow[name] + 
                    (1 - self.decay) * param.data
                )
                
    def apply_shadow(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data.clone()
                param.data = self.shadow[name]
                
    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data = self.backup[name]
        self.backup = {}


class LabelSmoothingCrossEntropy(nn.Module):
    """Cross-entropy avec label smoothing."""
    
    def __init__(self, smoothing: float = 0.1, reduction: str = 'mean'):
        super().__init__()
        self.smoothing = smoothing
        self.reduction = reduction
        
    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        n_classes = logits.size(-1)
        log_probs = F.log_softmax(logits, dim=-1)
        
        # Create smoothed labels
        with torch.no_grad():
            true_dist = torch.zeros_like(log_probs)
            true_dist.fill_(self.smoothing / (n_classes - 1))
            true_dist.scatter_(1, target.unsqueeze(1), 1.0 - self.smoothing)
        
        loss = (-true_dist * log_probs).sum(dim=-1)
        
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss

In [ ]:
# ==============================================================================
# TRAINING FUNCTIONS
# ==============================================================================

def train_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    scheduler: Optional[object] = None,
    scaler: Optional[GradScaler] = None,
    ema: Optional[EMA] = None,
    clip_grad: float = 1.0,
    device: torch.device = device
) -> Tuple[float, float, float]:
    """Entraîne le modèle pour une epoch."""
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []
    
    pbar = tqdm(loader, desc="Training", leave=False)
    for batch in pbar:
        tweet, user, meta, labels = [x.to(device) for x in batch]
        
        optimizer.zero_grad()
        
        # Mixed precision forward
        if scaler is not None:
            with autocast():
                logits = model(tweet, user, meta)
                loss = criterion(logits, labels)
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(tweet, user, meta)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            optimizer.step()
        
        if scheduler is not None:
            scheduler.step()
            
        if ema is not None:
            ema.update()
        
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    
    return avg_loss, acc, f1


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device = device
) -> Tuple[float, float, float, np.ndarray]:
    """Évalue le modèle."""
    model.eval()
    total_loss = 0
    all_preds, all_labels, all_probs = [], [], []
    
    for batch in tqdm(loader, desc="Evaluating", leave=False):
        tweet, user, meta, labels = [x.to(device) for x in batch]
        
        logits = model(tweet, user, meta)
        loss = criterion(logits, labels)
        
        total_loss += loss.item()
        probs = F.softmax(logits, dim=1).cpu().numpy()
        preds = np.argmax(probs, axis=1)
        
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        all_probs.append(probs)
    
    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    probs = np.vstack(all_probs)
    
    return avg_loss, acc, f1, probs

In [ ]:
# ==============================================================================
# CROSS-VALIDATION TRAINING
# ==============================================================================

def train_with_cv(
    X_tweet: np.ndarray,
    X_user: np.ndarray,
    X_meta: np.ndarray,
    y: np.ndarray,
    config: TrainingConfig
) -> Tuple[List[nn.Module], np.ndarray]:
    """Entraînement avec cross-validation stratifiée."""
    
    kfold = StratifiedKFold(n_splits=config.n_folds, shuffle=True, random_state=SEED)
    oof_preds = np.zeros((len(y), n_classes))
    models = []
    fold_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(kfold.split(X_tweet, y)):
        print(f"\n{'='*60}")
        print(f"📂 Fold {fold + 1}/{config.n_folds}")
        print(f"{'='*60}")
        
        # Split data
        X_t_train, X_t_val = X_tweet[train_idx], X_tweet[val_idx]
        X_u_train, X_u_val = X_user[train_idx], X_user[val_idx]
        X_m_train, X_m_val = X_meta[train_idx], X_meta[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        
        # Create datasets
        train_dataset = InfluencerDataset(X_t_train, X_u_train, X_m_train, y_train)
        val_dataset = InfluencerDataset(X_t_val, X_u_val, X_m_val, y_val)
        
        train_loader = DataLoader(
            train_dataset, 
            batch_size=config.batch_size, 
            shuffle=True, 
            num_workers=2, 
            pin_memory=True
        )
        val_loader = DataLoader(
            val_dataset, 
            batch_size=config.batch_size * 2, 
            shuffle=False, 
            num_workers=2, 
            pin_memory=True
        )
        
        # Initialize model
        model = ImprovedInfluencerModel(
            tweet_dim, user_dim, meta_dim, n_classes,
            dropout=config.dropout
        ).to(device)
        
        # Loss and optimizer
        criterion = LabelSmoothingCrossEntropy(smoothing=config.label_smoothing)
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=config.learning_rate,
            weight_decay=config.weight_decay
        )
        
        # Scheduler
        total_steps = len(train_loader) * config.epochs
        scheduler = OneCycleLR(
            optimizer,
            max_lr=config.learning_rate,
            total_steps=total_steps,
            pct_start=0.1,
            anneal_strategy='cos'
        )
        
        # AMP scaler
        scaler = GradScaler() if config.use_amp and device.type == 'cuda' else None
        
        # EMA
        ema = EMA(model, decay=config.ema_decay) if config.use_ema else None
        
        # Early stopping
        early_stopping = EarlyStopping(patience=config.patience, min_delta=config.min_delta)
        
        # Training loop
        best_f1 = 0
        best_state = None
        
        for epoch in range(1, config.epochs + 1):
            train_loss, train_acc, train_f1 = train_epoch(
                model, train_loader, optimizer, criterion,
                scheduler=scheduler, scaler=scaler, ema=ema,
                clip_grad=config.gradient_clip
            )
            
            # Use EMA weights for evaluation
            if ema is not None:
                ema.apply_shadow()
            
            val_loss, val_acc, val_f1, val_probs = evaluate(
                model, val_loader, criterion
            )
            
            if ema is not None:
                ema.restore()
            
            print(f"  Epoch {epoch:2d}/{config.epochs} | "
                  f"Train: loss={train_loss:.4f}, acc={train_acc:.4f}, f1={train_f1:.4f} | "
                  f"Val: loss={val_loss:.4f}, acc={val_acc:.4f}, f1={val_f1:.4f}")
            
            # Save best model
            if val_f1 > best_f1:
                best_f1 = val_f1
                if ema is not None:
                    ema.apply_shadow()
                best_state = copy.deepcopy(model.state_dict())
                if ema is not None:
                    ema.restore()
                print(f"  ✅ New best F1: {best_f1:.4f}")
            
            # Early stopping
            if early_stopping(val_f1):
                print(f"  ⏹️ Early stopping at epoch {epoch}")
                break
        
        # Load best weights
        model.load_state_dict(best_state)
        
        # Get OOF predictions
        model.eval()
        _, _, _, val_probs = evaluate(model, val_loader, criterion)
        oof_preds[val_idx] = val_probs
        
        # Save fold model
        fold_path = os.path.join(config.model_dir, f'model_fold{fold}.pt')
        torch.save({
            'model_state_dict': model.state_dict(),
            'fold': fold,
            'best_f1': best_f1,
            'config': {
                'tweet_dim': tweet_dim,
                'user_dim': user_dim,
                'meta_dim': meta_dim,
                'n_classes': n_classes
            }
        }, fold_path)
        
        models.append(model)
        fold_scores.append(best_f1)
        
        # Cleanup
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()
    
    # Summary
    print(f"\n{'='*60}")
    print("📊 CROSS-VALIDATION SUMMARY")
    print(f"{'='*60}")
    for i, score in enumerate(fold_scores):
        print(f"  Fold {i+1}: F1 = {score:.4f}")
    print(f"  Mean F1: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")
    
    # OOF metrics
    oof_labels = np.argmax(oof_preds, axis=1)
    oof_f1 = f1_score(y, oof_labels, average='macro')
    oof_acc = accuracy_score(y, oof_labels)
    print(f"\n  OOF F1: {oof_f1:.4f}")
    print(f"  OOF Accuracy: {oof_acc:.4f}")
    
    return models, oof_preds

In [ ]:
# ==============================================================================
# RUN TRAINING
# ==============================================================================

if config.use_cv:
    print("🚀 Starting Cross-Validation Training...")
    models, oof_preds = train_with_cv(X_train_t, X_train_u, X_train_meta, y, config)
    
    # Save OOF predictions
    np.save(os.path.join(config.model_dir, 'oof_predictions.npy'), oof_preds)
else:
    print("🚀 Starting Single-Fold Training...")
    # Simple train/val split
    X_t_train, X_t_val, X_u_train, X_u_val, X_m_train, X_m_val, y_train, y_val = train_test_split(
        X_train_t, X_train_u, X_train_meta, y,
        test_size=0.15, random_state=SEED, stratify=y
    )
    
    # Training code here (similar to fold training)
    # ...

In [ ]:
# ==============================================================================
# INFERENCE ON KAGGLE TEST SET
# ==============================================================================

@torch.no_grad()
def predict_ensemble(
    models: List[nn.Module],
    X_tweet: np.ndarray,
    X_user: np.ndarray,
    X_meta: np.ndarray,
    batch_size: int = 128
) -> np.ndarray:
    """Prédiction ensemble avec moyenne des probabilités."""
    
    dataset = InfluencerDataset(X_tweet, X_user, X_meta)
    loader = DataLoader(
        dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=2, 
        pin_memory=True
    )
    
    all_probs = []
    
    for model in models:
        model.eval()
        model_probs = []
        
        for batch in tqdm(loader, desc="Predicting", leave=False):
            tweet, user, meta = [x.to(device) for x in batch]
            logits = model(tweet, user, meta)
            probs = F.softmax(logits, dim=1).cpu().numpy()
            model_probs.append(probs)
        
        all_probs.append(np.vstack(model_probs))
    
    # Average probabilities
    avg_probs = np.mean(all_probs, axis=0)
    return avg_probs

print("\n🔮 Generating predictions on Kaggle test set...")
kaggle_probs = predict_ensemble(models, X_kaggle_t, X_kaggle_u, X_kaggle_meta)
kaggle_preds = np.argmax(kaggle_probs, axis=1)

print(f"   Predictions shape: {kaggle_preds.shape}")
print(f"   Class distribution: {np.bincount(kaggle_preds)}")

In [ ]:
# ==============================================================================
# CREATE SUBMISSION
# ==============================================================================

# Load Kaggle test IDs
kaggle_df = pd.read_json(os.path.join(config.data_dir, 'kaggle_test.jsonl'), lines=True)
kaggle_df = pd.json_normalize(kaggle_df.to_dict(orient='records'))

if 'challenge_id' in kaggle_df.columns:
    ids = kaggle_df['challenge_id'].astype(int).values
else:
    ids = np.arange(len(kaggle_preds))

# Create submission DataFrame
submission = pd.DataFrame({
    'ID': ids,
    'label': kaggle_preds
})

# Save submission
submission_path = os.path.join(config.submission_dir, 'submission_nn_improved.csv')
submission.to_csv(submission_path, index=False)

print(f"\n✅ Submission saved to {submission_path}")
print(f"   Total predictions: {len(submission)}")
print(f"\n📊 Prediction distribution:")
print(submission['label'].value_counts())

submission.head(10)

In [ ]:
# ==============================================================================
# SAVE PROBABILITIES FOR ENSEMBLE
# ==============================================================================

# Save probabilities for later ensemble with other models
np.save(os.path.join(config.model_dir, 'kaggle_nn_probs.npy'), kaggle_probs)
print(f"\n💾 Saved Kaggle probabilities for ensemble: {kaggle_probs.shape}")

# Summary
print("\n" + "="*60)
print("✅ TRAINING COMPLETE")
print("="*60)
print(f"\n📁 Saved files:")
print(f"   - {config.n_folds} model checkpoints")
print(f"   - OOF predictions")
print(f"   - Kaggle probabilities")
print(f"   - Submission CSV")